In [21]:
import torch
from torch import nn
import subprocess
import os
import numpy as np
from tqdm import tqdm
from SSM_driver import LegMeasurementDataset, measure


In [ ]:
config = {
    "lr": 1e-4,
    "eta_min": 1e-8,
    "batch_size": 16,
    "log": True,
    "seed": 42,
    "epochs":250,
}
dtype = torch.float64

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = LegMeasurementDataset(measure, batch_size=config["batch_size"], dtype=dtype, device=device)
dataloader = torch.utils.data.DataLoader(dataset, config["batch_size"], shuffle=False, num_workers=0)
component_transforms = torch.load("./data_components/scaled_component_transforms.pt").to(dtype).to(device)

print(component_transforms.shape)



torch.Size([2, 11])


In [27]:
def step(x, y, func, lambda_=1E-3, delta=1E-4):
    # x.shape = (batch, components)
    # func(x).shape = (batch, measures)
    # y.shape = (batch, measures)
    with torch.no_grad():
        deltas = torch.eye(x.shape[1], device=x.device, dtype=x.dtype)*delta
        pos = func((x[:, None] + deltas).reshape((x.shape[0]*x.shape[1], x.shape[1]))).reshape((x.shape[0], y.shape[1], x.shape[1]))
        neg = func((x[:, None] - deltas).reshape((x.shape[0]*x.shape[1], x.shape[1]))).reshape((x.shape[0], y.shape[1], x.shape[1]))
        func_x = func(x)

        J_approx = -(pos - neg) / (2*delta)

        # Compute residuals: (batch_size, n_measures, 1)
        residuals = (y - func_x).unsqueeze(-1)

        # JTJ: (batch_size, n_components, n_components)
        JTJ = torch.bmm(J_approx.transpose(1, 2), J_approx)

        # Damping (use scalar, e.g. 1e-3)
        damping = torch.diag_embed(torch.diagonal(JTJ, dim1=1, dim2=2))

        # J^T * residuals: (batch_size, n_components, 1)
        JTr = torch.bmm(J_approx.transpose(1, 2), residuals)

        # LM step direction: (batch_size, n_components)
        x_new = x + torch.linalg.solve(JTJ + lambda_ * damping, JTr).squeeze(-1)
        new_residuals = (y - func(x_new))
        while torch.sum(residuals**2) < torch.sum(new_residuals**2):
            lambda_ *= 10
            x_new = x + torch.linalg.solve(JTJ + lambda_ * damping, JTr).squeeze(-1)
            new_residuals = (y - func(x_new))

            if lambda_ > 1E10:
                raise ValueError()

        return x_new, new_residuals, lambda_ / 10


for measures, components in dataloader:
    # Mean values
    x = torch.normal(torch.zeros_like(components), torch.ones_like(components)) * component_transforms[1:] + component_transforms[:1]
    lambda_ = 1E-20
    for _ in range(20):
        (x, residuals, lambda_), x_old = step(x, measures, dataset.get_measures, lambda_), x
        print(torch.max(torch.abs(residuals), dim=-1)[0], lambda_, residuals.shape)
    break



tensor([133.8252], dtype=torch.float64) 1.0000000000000001e-16 torch.Size([1, 7])


ValueError: 

In [ ]:
from nb_exporter import export_notebook
export_notebook("LMOptimisationAttempt.ipynb")
export_notebook("SSM_driver.ipynb")